In [1]:
import sqlite3
import pandas as pd
import platform
import warnings
warnings.filterwarnings('ignore')

# Detecção do SO e caminho BD
if platform.system() == 'Windows':
    DB_PATH = r"C:\Users\LISARR\Documents\python\01.Financeiro\inform_27.db"
elif platform.system() == 'Darwin':
    DB_PATH = "/Volumes/RR/DB/inform_27.db"
else:
    DB_PATH = "inform_27.db"

# Conectar
con = sqlite3.connect(DB_PATH)
cursor = con.cursor()

In [2]:
df = pd.read_sql_query("SELECT * FROM inform_27_2026", con)
con.close()

In [3]:
print(f"Shape: {df.shape}")
print(f"\nColunas: {df.shape[1]}")
df.head(1)

Shape: (108219, 58)

Colunas: 58


,id,PROPIETARIO,TRAYECTO,TRANSPORTISTA,TRACTORA,REMOLQUE,INGRESODT,COSTEDT,RENTADT,PALETSDT,...,LOCDES,LUGARDESCARGA,TEMP_MERC_PED,TIPOPALETA,CAMION_TIPO,CAMION_CAPACIDAD,TIPO_COMBUSTIBLE,KMREALES,ALBARAN,ficheiro_origem
0,1,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,SOLIDO CLICK LDA,0000XXX,0000XXX,10.69,4.64,6.05,0.49,...,573897,"PLAT. COOPLECNORTE,S.A.",RFGRFG,EUR,Trailer 33 plts,33,DIESEL,194.134,NO,SAL_DAT027 (11).xls


In [4]:
# Converter tipos de dados numéricos
df['INGRESODT'] = pd.to_numeric(df['INGRESODT'], errors='coerce')
df['COSTEDT'] = pd.to_numeric(df['COSTEDT'], errors='coerce')
df['RENTADT'] = pd.to_numeric(df['RENTADT'], errors='coerce')
df['PALETSDT'] = pd.to_numeric(df['PALETSDT'], errors='coerce')
df['PESO_BRUTO'] = pd.to_numeric(df['PESO_BRUTO'], errors='coerce')
df['PALETS'] = pd.to_numeric(df['PALETS'], errors='coerce')
df['KM'] = pd.to_numeric(df['KM'], errors='coerce')
df['KMREALES'] = pd.to_numeric(df['KMREALES'], errors='coerce')

# CODEUT como string
df['CODEUT'] = df['CODEUT'].astype(str)

# Trim em strings (ANTES de converter as datas)
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

# Datas como datetime (sem hora, prontas para PBI) - DEPOIS do trim
df['FCARGA'] = pd.to_datetime(df['FCARGA'], format='%Y%m%d', errors='coerce').dt.date
df['FENTREGA'] = pd.to_datetime(df['FENTREGA'], format='%Y%m%d', errors='coerce').dt.date

In [5]:
print(f"Shape: {df.shape}")
print(f"\nTipos de dados:")
print(df.dtypes)
print(f"\nNulos por coluna:")
print(df.isnull().sum())


Shape: (108219, 58)

Tipos de dados:
id                    int64
PROPIETARIO             str
TRAYECTO                str
TRANSPORTISTA           str
TRACTORA                str
REMOLQUE                str
INGRESODT           float64
COSTEDT             float64
RENTADT             float64
PALETSDT            float64
PESO_BRUTO          float64
CODEUT                  str
ESTADO_UT               str
RANGO_UT                str
FCARGA               object
ACTIVIDAD               str
CODEDT                  str
ESTADO_DT               str
REFERENCIA              str
CODACT                  str
LOCORIGEN               str
PROV_ORIGEN             str
PAISORIGEN              str
CPOSTAL                 str
LOCDESTINO              str
PROV_DESTINO            str
PAISDESTINO             str
CPOSTAD                 str
KM                  float64
FENTREGA             object
ORIGEN                  str
ENTREGAR                str
PROV_ENTREGAR           str
PAISENTREGAR            str
DESTINO    

In [6]:
df.head(5)

,id,PROPIETARIO,TRAYECTO,TRANSPORTISTA,TRACTORA,REMOLQUE,INGRESODT,COSTEDT,RENTADT,PALETSDT,...,LOCDES,LUGARDESCARGA,TEMP_MERC_PED,TIPOPALETA,CAMION_TIPO,CAMION_CAPACIDAD,TIPO_COMBUSTIBLE,KMREALES,ALBARAN,ficheiro_origem
0,1,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,SOLIDO CLICK LDA,0000XXX,0000XXX,10.69,4.64,6.05,0.49,...,573897,"PLAT. COOPLECNORTE,S.A.",RFGRFG,EUR,Trailer 33 plts,33,DIESEL,194.134,NO,SAL_DAT027 (11).xls
1,2,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,SOLIDO CLICK LDA,0000XXX,0000XXX,125.04,236.86,-111.82,25.00,...,573897,"PLAT. COOPLECNORTE,S.A.",RFGRFG,EUR,Trailer 33 plts,33,DIESEL,194.134,NO,SAL_DAT027 (11).xls
2,3,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,0000XXX,0000XXX,0.00,12.02,-12.02,0.80,...,298515,DL Transportes Fernando Simões Monteiro Unip. Lda,RFGRFG,EUR,Trailer 33 plts,33,DIESEL,151.803,NO,SAL_DAT027 (11).xls
3,4,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,0000XXX,0000XXX,0.00,10.82,-10.82,0.72,...,298515,DL Transportes Fernando Simões Monteiro Unip. Lda,RFGRFG,EUR,Trailer 33 plts,33,DIESEL,151.803,NO,SAL_DAT027 (11).xls
4,5,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,0000XXX,0000XXX,0.00,6.01,-6.01,0.40,...,298515,DL Transportes Fernando Simões Monteiro Unip. Lda,RFGRFG,EUR,Trailer 33 plts,33,DIESEL,151.803,NO,SAL_DAT027 (11).xls


CUSTOS DANONE

In [7]:
# Caminho do ficheiro
caminho = r"C:\Users\LISARR\Documents\python\01.Financeiro\Danone_Custos_V2.xlsx"

# Ler a dinâmica a partir da linha 7 (skiprows=6), até coluna N
df_danone = pd.read_excel(
    caminho,
    sheet_name="Ingreso_coste_RR",
    skiprows=6,  # Pula as primeiras 6 linhas para começar na linha 7
    usecols=range(14)  # Colunas A-N (0-13)
)

# Remover a última linha se for totais (verifica o padrão)
if df_danone.iloc[-1, 0] in ['Total', 'TOTAL', 'Totals']:
    df_danone = df_danone[:-1]

# Garantir tipos de dados apropriados onde necessário
df_danone = df_danone.reset_index(drop=True)

In [8]:
print(f"Removidas {(df_danone['CODEDT'].isna() | (df_danone['CODEDT'] == '(blank)')).sum()} linhas"); df_danone = df_danone[~((df_danone['CODEDT'].isna()) | (df_danone['CODEDT'] == '(blank)'))]

Removidas 2 linhas


In [9]:
df_danone.head(5)

,CODEDT,FENTREGA,REFERENCIA,CODEUT,PEDPESONETO,LOCORIGEN,LOCDESTINO,PROV_DESTINO,Suma de COSTEDT,Sum of Ingreso,Resultado,Grupo tarifario,Month,Unnamed: 13
0,25392905,2026-01-02,5021755057 413962691,3347518.0,8.35,CMTIR,A.S. FRANCOS,Porto,4.04,30.843115,26.803115,Ofertas,1,NaN
1,25392907,2026-01-02,5021735096 413969126,3347519.0,4.50,CMTIR,A.S. SRA. DA HORA,Porto,0.50,30.843115,30.343115,Ofertas,1,NaN
2,25392909,2026-01-02,5021755060 413965689,3347519.0,6.08,CMTIR,A.S. BOAVISTA - DISTRIBUIÇÃO NORMAL,Porto,0.50,30.843115,30.343115,Ofertas,1,NaN
3,25392911,2026-01-02,5021721307 413942431,3347512.0,32.04,CMTIR,PLATAFORMA DO PORTO - 2101,Porto,4.68,0.909688,-3.770312,GS Centro,1,NaN
4,25393080,2026-01-06,5021741331 413966458,3346529.0,2.16,SALVESEN LOGISTICA PORTUGAL,OFERTA PRODUTO -R.HUMANOS,Lisboa,0.41,0.000000,-0.410000,Sin Transporte Marketing,1,NaN


In [10]:
# Preparar df_danone - selecionar colunas necessárias
df_danone = df_danone[['CODEDT', 'FENTREGA', 'CODEUT', 'Sum of Ingreso']].copy()
df_danone.columns = ['CODEDT', 'FENTREGA', 'CODEUT', 'Sum of Ingreso']

# Converter tipos de dados - df_danone
df_danone['CODEDT'] = pd.to_numeric(df_danone['CODEDT'], errors='coerce').fillna(0).astype('Int64')
df_danone['CODEUT'] = pd.to_numeric(df_danone['CODEUT'], errors='coerce').astype('Int64').astype(str).str.strip()
df_danone['FENTREGA'] = pd.to_datetime(df_danone['FENTREGA']).dt.strftime('%Y%m%d')
df_danone['Sum of Ingreso'] = pd.to_numeric(df_danone['Sum of Ingreso'], errors='coerce')

# Limpeza no df - MESMO TRATAMENTO (FENTREGA sem hífens)
df['CODEUT'] = pd.to_numeric(df['CODEUT'], errors='coerce').astype('Int64').astype(str).str.strip()
df['CODEDT'] = pd.to_numeric(df['CODEDT'], errors='coerce').fillna(0).astype('Int64')
df['FENTREGA'] = df['FENTREGA'].astype(str).str.replace('-', '', regex=False).str.strip()

# Remover coluna antiga 'ingresso_danone' se já existir (de execuções anteriores)
if 'ingresso_danone' in df.columns:
    df = df.drop(columns=['ingresso_danone'])

# Verificar duplicados em df_danone por CODEDT + FENTREGA antes do merge
dup = df_danone.duplicated(subset=['CODEDT', 'FENTREGA'], keep=False)
print(f"Linhas duplicadas em df_danone (CODEDT+FENTREGA): {dup.sum()}")

# Match por CODEDT + FENTREGA (merge)
df = df.merge(
    df_danone[['CODEDT', 'FENTREGA', 'Sum of Ingreso']],
    on=['CODEDT', 'FENTREGA'],
    how='left'
)
df = df.rename(columns={'Sum of Ingreso': 'ingresso_danone'})

# Estatísticas
total_linhas = len(df)
matches = df['ingresso_danone'].notna().sum()
sum_ingresso_danone = df['ingresso_danone'].sum()
sum_ingreso_excel = df_danone['Sum of Ingreso'].sum()

print(f"Total de linhas df: {total_linhas}")
print(f"Matches encontrados: {matches}")
print(f"Taxa sucesso: {(matches/total_linhas*100):.2f}%")
print(f"\nSum ingresso_danone: {sum_ingresso_danone:,.2f}")
print(f"Sum Ingreso Excel: {sum_ingreso_excel:,.2f}")
print(f"Diferença: {(sum_ingreso_excel - sum_ingresso_danone):,.2f}")

df[['CODEDT', 'FENTREGA', 'CODEUT', 'INGRESODT', 'ingresso_danone']].head(10)

Linhas duplicadas em df_danone (CODEDT+FENTREGA): 0
Total de linhas df: 108219
Matches encontrados: 16491
Taxa sucesso: 15.24%

Sum ingresso_danone: 1,731,835.73
Sum Ingreso Excel: 1,731,835.73
Diferença: -0.00


,CODEDT,FENTREGA,CODEUT,INGRESODT,ingresso_danone
0,25387086,20260102,3344051,10.69,NaN
1,25398553,20260102,3344051,125.04,NaN
2,25392919,20260102,3344052,0.00,NaN
3,25393741,20260102,3344052,0.00,NaN
4,25393743,20260102,3344052,0.00,NaN
5,25392920,20260102,3344163,0.00,NaN
6,25393742,20260102,3344163,0.00,NaN
7,25393744,20260102,3344163,0.00,NaN
8,25391965,20260102,3344178,23.54,NaN
9,25394896,20260102,3344178,15.07,NaN


In [11]:
# Escolher um exemplo concreto de df_danone
linha_teste = df_danone.iloc[15]
codedt_teste = linha_teste['CODEDT']
fentrega_teste = linha_teste['FENTREGA']

print(f"CODEDT: {codedt_teste}")
print(f"FENTREGA: {fentrega_teste}")

print("\nValor no df_danone:")
print(df_danone[(df_danone['CODEDT'] == codedt_teste) & (df_danone['FENTREGA'] == fentrega_teste)][['CODEDT', 'FENTREGA', 'Sum of Ingreso']])

print("\nValor no df (coluna ingresso_danone, depois do merge):")
print(df[(df['CODEDT'] == codedt_teste) & (df['FENTREGA'] == fentrega_teste)][['CODEDT', 'FENTREGA', 'ingresso_danone']])

CODEDT: 25395002
FENTREGA: 20260102

Valor no df_danone:
      CODEDT  FENTREGA  Sum of Ingreso
15  25395002  20260102       26.897032

Valor no df (coluna ingresso_danone, depois do merge):
       CODEDT  FENTREGA  ingresso_danone
586  25395002  20260102        26.897032


In [12]:
df.head(5)

,id,PROPIETARIO,TRAYECTO,TRANSPORTISTA,TRACTORA,REMOLQUE,INGRESODT,COSTEDT,RENTADT,PALETSDT,...,LUGARDESCARGA,TEMP_MERC_PED,TIPOPALETA,CAMION_TIPO,CAMION_CAPACIDAD,TIPO_COMBUSTIBLE,KMREALES,ALBARAN,ficheiro_origem,ingresso_danone
0,1,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,SOLIDO CLICK LDA,0000XXX,0000XXX,10.69,4.64,6.05,0.49,...,"PLAT. COOPLECNORTE,S.A.",RFGRFG,EUR,Trailer 33 plts,33,DIESEL,194.134,NO,SAL_DAT027 (11).xls,NaN
1,2,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,SOLIDO CLICK LDA,0000XXX,0000XXX,125.04,236.86,-111.82,25.00,...,"PLAT. COOPLECNORTE,S.A.",RFGRFG,EUR,Trailer 33 plts,33,DIESEL,194.134,NO,SAL_DAT027 (11).xls,NaN
2,3,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,0000XXX,0000XXX,0.00,12.02,-12.02,0.80,...,DL Transportes Fernando Simões Monteiro Unip. Lda,RFGRFG,EUR,Trailer 33 plts,33,DIESEL,151.803,NO,SAL_DAT027 (11).xls,NaN
3,4,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,0000XXX,0000XXX,0.00,10.82,-10.82,0.72,...,DL Transportes Fernando Simões Monteiro Unip. Lda,RFGRFG,EUR,Trailer 33 plts,33,DIESEL,151.803,NO,SAL_DAT027 (11).xls,NaN
4,5,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,0000XXX,0000XXX,0.00,6.01,-6.01,0.40,...,DL Transportes Fernando Simões Monteiro Unip. Lda,RFGRFG,EUR,Trailer 33 plts,33,DIESEL,151.803,NO,SAL_DAT027 (11).xls,NaN


Calculos auxiliares

In [13]:
# Criar novas colunas antes de exportar para parquet

# Converter FCARGA para datetime para extrair informações
df['data'] = pd.to_datetime(df['FCARGA'])

# Week number (ISO week)
df['week_number'] = df['data'].dt.isocalendar().week

# Week day (0=Segunda, 6=Domingo)
df['week_day'] = df['data'].dt.day_name()

# Mês e nome do mês (a partir de FENTREGA, já normalizado para formato YYYYMMDD)
df['mes'] = pd.to_datetime(df['FENTREGA'], format='%Y%m%d').dt.month
df['mes_nome'] = pd.to_datetime(df['FENTREGA'], format='%Y%m%d').dt.strftime('%B')

# Total ingresso = ingresso_danone + INGRESODT (soma direta, sem /2 - já validado com merge por CODEDT+FENTREGA)
df['total_ingresso'] = df['ingresso_danone'].fillna(0)/2 + df['INGRESODT']

print("Colunas criadas:")
print(f"  ✓ week_number")
print(f"  ✓ week_day")
print(f"  ✓ mes")
print(f"  ✓ mes_nome")
print(f"  ✓ total_ingresso")

print(f"\nExemplos:")
df[['FCARGA', 'week_number', 'week_day', 'mes', 'mes_nome', 'INGRESODT', 'ingresso_danone', 'total_ingresso']].head(10)

Colunas criadas:
  ✓ week_number
  ✓ week_day
  ✓ mes
  ✓ mes_nome
  ✓ total_ingresso

Exemplos:


,FCARGA,week_number,week_day,mes,mes_nome,INGRESODT,ingresso_danone,total_ingresso
0,2025-12-29,1,Monday,1,January,10.69,NaN,10.69
1,2025-12-29,1,Monday,1,January,125.04,NaN,125.04
2,2025-12-30,1,Tuesday,1,January,0.00,NaN,0.00
3,2025-12-30,1,Tuesday,1,January,0.00,NaN,0.00
4,2025-12-30,1,Tuesday,1,January,0.00,NaN,0.00
5,2025-12-30,1,Tuesday,1,January,0.00,NaN,0.00
6,2025-12-30,1,Tuesday,1,January,0.00,NaN,0.00
7,2025-12-30,1,Tuesday,1,January,0.00,NaN,0.00
8,2026-01-02,1,Friday,1,January,23.54,NaN,23.54
9,2026-01-02,1,Friday,1,January,15.07,NaN,15.07


tratamento antes das metricas

In [14]:
# Garantir que capacidade_norm existe
if 'capacidade_norm' not in df.columns:
    print("Criando capacidade_norm...")
    
    # Converter para número
    df['CAMION_CAPACIDAD_NUM'] = pd.to_numeric(df['CAMION_CAPACIDAD'], errors='coerce')
    
    # Mapeamento de normalização
    mapeamento = {
        4: 6, 5: 6, 6: 6,
        8: 12, 12: 12,
        14: 20, 15: 20, 16: 20, 18: 20, 20: 20,
        22: 24, 24: 24,
        33: 33, 66: 66
    }
    
    # Aplicar mapeamento
    df['capacidade_norm'] = df['CAMION_CAPACIDAD_NUM'].map(mapeamento)
    
    # Tratar zeros
    linhas_zero = df['CAMION_CAPACIDAD_NUM'] == 0
    if linhas_zero.sum() > 0:
        capacidade_por_codeut = df[df['CAMION_CAPACIDAD_NUM'] > 0].groupby('CODEUT')['capacidade_norm'].agg(lambda x: x.mode()[0] if len(x.mode()) > 0 else x.max())
        for cu in df[linhas_zero]['CODEUT'].unique():
            if cu in capacidade_por_codeut.index:
                df.loc[(df['CODEUT'] == cu) & linhas_zero, 'capacidade_norm'] = capacidade_por_codeut[cu]

# Criar coluna indicadora
df['dados_validos'] = df['capacidade_norm'].notna()

print("="*80)
print("MARCAÇÃO DE DADOS")
print("="*80)

print(f"Total linhas: {len(df):,}")
print(f"Dados válidos: {df['dados_validos'].sum():,}")
print(f"Dados incompletos (ignorar): {(~df['dados_validos']).sum():,}")

print(f"\n--- Cálculos usarão apenas linhas com dados_validos = True ---")

Criando capacidade_norm...
MARCAÇÃO DE DADOS
Total linhas: 108,219
Dados válidos: 107,958
Dados incompletos (ignorar): 261

--- Cálculos usarão apenas linhas com dados_validos = True ---


Coordenadas

In [15]:
# Ler ficheiro de coordenadas
import platform
if platform.system() == 'Windows':
    coords_path = r"C:\Users\LISARR\Documents\python\00.Metadados\CP7COORDS_FORMATADO.CSV"
elif platform.system() == 'Darwin':
    coords_path = "/Volumes/RR/DB/CP7COORDS_FORMATADO.CSV"
else:
    coords_path = "CP7COORDS_FORMATADO.CSV"

coords = pd.read_csv(coords_path, sep=';', encoding='utf-8')
coords['POINT_X'] = coords['POINT_X'].str.replace(',', '.').astype(float)
coords['POINT_Y'] = coords['POINT_Y'].str.replace(',', '.').astype(float)

print(f"Coordenadas carregadas: {len(coords):,} registos")



Coordenadas carregadas: 178,409 registos


In [16]:
# Calcular centroid por primeira parte do CP
coords['CP_parte'] = coords['CP'].str.split('-').str[0]
coords_centroid = coords.groupby('CP_parte')[['POINT_X', 'POINT_Y']].mean().reset_index()
coords_centroid.columns = ['CP_parte', 'longitude_centroid', 'latitude_centroid']

coords_centroid.head()

,CP_parte,longitude_centroid,latitude_centroid
0,1000,-9.139397,38.738262
1,1050,-9.149352,38.734860
2,1070,-9.164875,38.729817
3,1100,-9.131875,38.713056
4,1150,-9.140143,38.722876


In [17]:
# Calcular centroid por primeira parte do CP
coords['CP_parte'] = coords['CP'].str.split('-').str[0]
centroid = coords.groupby('CP_parte')[['POINT_X', 'POINT_Y']].mean().reset_index()
centroid.columns = ['CP_parte', 'longitude_centroid', 'latitude_centroid']

# Preparar dados para origem
coords_origem = coords[['CP', 'POINT_X', 'POINT_Y']].copy()
coords_origem.columns = ['CPOSTAL', 'longitude_origem', 'latitude_origem']

# Preparar dados para destino
coords_destino = coords[['CP', 'POINT_X', 'POINT_Y']].copy()
coords_destino.columns = ['CPOSTAD', 'longitude_destino', 'latitude_destino']

# Remover colunas de coordenadas se já existem (de execuções anteriores)
cols_remover = ['longitude_origem', 'latitude_origem', 'longitude_destino', 'latitude_destino', 'CPOSTAL_parte', 'CPOSTAD_parte']
df = df.drop(columns=[c for c in cols_remover if c in df.columns])

df['CPOSTAL_parte'] = df['CPOSTAL'].str.split('-').str[0]
df['CPOSTAD_parte'] = df['CPOSTAD'].str.split('-').str[0]

# Join origem e destino (como estava)
df = df.merge(coords_origem, on='CPOSTAL', how='left')
df = df.merge(coords_destino, on='CPOSTAD', how='left')

# Preencher NaN com centroid origem
df = df.merge(centroid.rename(columns={'CP_parte': 'CPOSTAL_parte', 'longitude_centroid': 'longitude_origem_c', 'latitude_centroid': 'latitude_origem_c'}), on='CPOSTAL_parte', how='left')
df.loc[df['longitude_origem'].isna(), 'longitude_origem'] = df.loc[df['longitude_origem'].isna(), 'longitude_origem_c']
df.loc[df['latitude_origem'].isna(), 'latitude_origem'] = df.loc[df['latitude_origem'].isna(), 'latitude_origem_c']

# Preencher NaN com centroid destino
df = df.merge(centroid.rename(columns={'CP_parte': 'CPOSTAD_parte', 'longitude_centroid': 'longitude_destino_c', 'latitude_centroid': 'latitude_destino_c'}), on='CPOSTAD_parte', how='left')
df.loc[df['longitude_destino'].isna(), 'longitude_destino'] = df.loc[df['longitude_destino'].isna(), 'longitude_destino_c']
df.loc[df['latitude_destino'].isna(), 'latitude_destino'] = df.loc[df['latitude_destino'].isna(), 'latitude_destino_c']

# Remover colunas temporárias
df = df.drop(columns=['CPOSTAL_parte', 'CPOSTAD_parte', 'longitude_origem_c', 'latitude_origem_c', 'longitude_destino_c', 'latitude_destino_c'])

print(f"Matches origem: {df['longitude_origem'].notna().sum():,}")
print(f"Matches destino: {df['longitude_destino'].notna().sum():,}")

Matches origem: 107,266
Matches destino: 87,588


In [18]:
df.head()

,id,PROPIETARIO,TRAYECTO,TRANSPORTISTA,TRACTORA,REMOLQUE,INGRESODT,COSTEDT,RENTADT,PALETSDT,...,mes,mes_nome,total_ingresso,CAMION_CAPACIDAD_NUM,capacidade_norm,dados_validos,longitude_origem,latitude_origem,longitude_destino,latitude_destino
0,1,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,SOLIDO CLICK LDA,0000XXX,0000XXX,10.69,4.64,6.05,0.49,...,1,January,10.69,33,33.0,True,-8.920854,39.047635,-8.523268,40.512134
1,2,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,SOLIDO CLICK LDA,0000XXX,0000XXX,125.04,236.86,-111.82,25.00,...,1,January,125.04,33,33.0,True,-8.920854,39.047635,-8.523268,40.512134
2,3,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,0000XXX,0000XXX,0.00,12.02,-12.02,0.80,...,1,January,0.00,33,33.0,True,-8.920854,39.047635,-8.531480,40.167186
3,4,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,0000XXX,0000XXX,0.00,10.82,-10.82,0.72,...,1,January,0.00,33,33.0,True,-8.920854,39.047635,-8.531480,40.167186
4,5,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,0000XXX,0000XXX,0.00,6.01,-6.01,0.40,...,1,January,0.00,33,33.0,True,-8.920854,39.047635,-8.531480,40.167186


In [19]:
# Teste: filtrar por CODEUT específico
codeut_teste = df['CODEUT'].iloc[0]  # substituir por um CODEUT específico se quiser

filtro = df[df['CODEUT'] == codeut_teste][['CODEUT', 'INGRESODT', 'ingresso_danone','total_ingresso']]
filtro

,CODEUT,INGRESODT,ingresso_danone,total_ingresso
0,3344051,10.69,NaN,10.69
1,3344051,125.04,NaN,125.04
83378,3344051,117.22,NaN,117.22


Cálculos

In [20]:
# Configurar pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Taxa de ocupação - aditiva por natureza (SUM das linhas no PBI = taxa da rota)
df['taxa_ocupacao'] = (df['PALETS'] / df['capacidade_norm'].replace(0, 1)) * 100

df[['CODEUT', 'PALETS', 'capacidade_norm', 'taxa_ocupacao']].head(5)

,CODEUT,PALETS,capacidade_norm,taxa_ocupacao
0,3344051,1.0,33.0,3.030303
1,3344051,25.0,33.0,75.757576
2,3344052,1.0,33.0,3.030303
3,3344052,1.0,33.0,3.030303
4,3344052,1.0,33.0,3.030303


In [21]:
# Lista de colunas finais
colunas_finais = [
    'FENTREGA', 'CODEUT', 'capacidade_norm', 'PALETS', 'INGRESODT', 'ingresso_danone', 'total_ingresso', 'COSTEDT',
    'PROV_ORIGEN', 'LOCORIGEN', 'PROV_DESTINO', 'LOCDESTINO', 'CPOSTAL', 'CPOSTAD',
    'TRANSPORTISTA', 'week_day', 'week_number', 'mes', 'mes_nome', 'data',
    'dados_validos', 'LOCCAR', 'LUGARCARGA', 'LOCDES', 'LUGARDESCARGA', 'TRACTORA','longitude_destino',	'latitude_destino',
    'PESO_BRUTO', 'CODEDT', 'CODACT', 'TIPOCLIENTE', 'TIPOFLUJO',
    'PROV_ENTREGAR', 'PAISENTREGAR', 'ACTIVIDAD',
    'taxa_ocupacao'
]

# Selecionar colunas
df_sel = df[colunas_finais].copy()

print(f"df_sel criado: {len(df_sel):,} linhas x {len(df_sel.columns)} colunas")
print(f"CPOSTAL únicos: {df_sel['CPOSTAL'].nunique()}")

df_sel criado: 108,219 linhas x 37 colunas
CPOSTAL únicos: 153


In [22]:
df_sel.head(5)

,FENTREGA,CODEUT,capacidade_norm,PALETS,INGRESODT,ingresso_danone,total_ingresso,COSTEDT,PROV_ORIGEN,LOCORIGEN,PROV_DESTINO,LOCDESTINO,CPOSTAL,CPOSTAD,TRANSPORTISTA,week_day,week_number,mes,mes_nome,data,dados_validos,LOCCAR,LUGARCARGA,LOCDES,LUGARDESCARGA,TRACTORA,longitude_destino,latitude_destino,PESO_BRUTO,CODEDT,CODACT,TIPOCLIENTE,TIPOFLUJO,PROV_ENTREGAR,PAISENTREGAR,ACTIVIDAD,taxa_ocupacao
0,20260102,3344051,33.0,1.0,10.69,NaN,10.69,4.64,Lisboa,Vila Nova da Rainha,Aveiro,Oliveira do Bairro,2050-540,3770-305,SOLIDO CLICK LDA,Monday,1,1,January,2025-12-29,True,137998,SALVESEN LOGISTICA PORTUGAL,573897,"PLAT. COOPLECNORTE,S.A.",0000XXX,-8.523268,40.512134,227.26,25387086,011,TLD PORTUGAL,Directo,Aveiro,PORTUGAL,DANONE PORTUGAL,3.030303
1,20260102,3344051,33.0,25.0,125.04,NaN,125.04,236.86,Lisboa,Vila Nova da Rainha,Aveiro,Oliveira do Bairro,2050-540,3770-305,SOLIDO CLICK LDA,Monday,1,1,January,2025-12-29,True,137998,SALVESEN LOGISTICA PORTUGAL,573897,"PLAT. COOPLECNORTE,S.A.",0000XXX,-8.523268,40.512134,2719.88,25398553,011,TLD PORTUGAL,Directo,Aveiro,PORTUGAL,DANONE PORTUGAL,75.757576
2,20260102,3344052,33.0,1.0,0.00,NaN,0.00,12.02,Lisboa,Vila Nova da Rainha,Coimbra,Condeixa-a-Nova,2050-540,3150-020,Transportes Fernando Simões Monteiro Unip. Lda,Tuesday,1,1,January,2025-12-30,True,137998,SALVESEN LOGISTICA PORTUGAL,298515,DL Transportes Fernando Simões Monteiro Unip. Lda,0000XXX,-8.531480,40.167186,487.26,25392919,011,TLD PORTUGAL,DLS,Aveiro,PORTUGAL,DANONE PORTUGAL,3.030303
3,20260102,3344052,33.0,1.0,0.00,NaN,0.00,10.82,Lisboa,Vila Nova da Rainha,Coimbra,Condeixa-a-Nova,2050-540,3150-020,Transportes Fernando Simões Monteiro Unip. Lda,Tuesday,1,1,January,2025-12-30,True,137998,SALVESEN LOGISTICA PORTUGAL,298515,DL Transportes Fernando Simões Monteiro Unip. Lda,0000XXX,-8.531480,40.167186,423.44,25393741,011,TLD PORTUGAL,DLS,Aveiro,PORTUGAL,DANONE PORTUGAL,3.030303
4,20260102,3344052,33.0,1.0,0.00,NaN,0.00,6.01,Lisboa,Vila Nova da Rainha,Coimbra,Condeixa-a-Nova,2050-540,3150-020,Transportes Fernando Simões Monteiro Unip. Lda,Tuesday,1,1,January,2025-12-30,True,137998,SALVESEN LOGISTICA PORTUGAL,298515,DL Transportes Fernando Simões Monteiro Unip. Lda,0000XXX,-8.531480,40.167186,236.29,25393743,011,TLD PORTUGAL,DLS,Leiria,PORTUGAL,DANONE PORTUGAL,3.030303


Exportar ficheiro


In [23]:
caminho_parquet = r"C:\Users\LISARR\Documents\python\01.Financeiro\inform_27_2026_final.parquet"

# Exportar para parquet
df_sel.to_parquet(caminho_parquet, index=False)

print(f"✅ Ficheiro exportado: {caminho_parquet}")
print(f"Linhas: {len(df_sel)}")
print(f"Colunas: {len(df_sel.columns)}")
print(f"Tamanho: {df_sel.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

✅ Ficheiro exportado: C:\Users\LISARR\Documents\python\01.Financeiro\inform_27_2026_final.parquet
Linhas: 108219
Colunas: 37
Tamanho: 53.20 MB



import sqlite3

# Conectar à BD
if platform.system() == 'Windows':
    DB_PATH = r"C:\Users\LISARR\Documents\python\01.Financeiro\inform_27.db"
elif platform.system() == 'Darwin':
    DB_PATH = "/Volumes/RR/DB/inform_27.db"
else:
    DB_PATH = "inform_27.db"

con = sqlite3.connect(DB_PATH)

# Ler tabela km_diario_2026
km_diario = pd.read_sql_query("SELECT * FROM km_diario_2026", con)
con.close()

print(f"km_diario_2026 carregada: {len(km_diario):,} linhas")

# Converter data para datetime
km_diario['data'] = pd.to_datetime(km_diario['data'])

# Renomear coluna trator para TRACTORA para match
km_diario.rename(columns={'trator': 'TRACTORA'}, inplace=True)

# Converter TRACTORA em df_sel para datetime (se necessário)
df_sel['data'] = pd.to_datetime(df_sel['data'])

# Join: data + TRACTORA
df_merged = df_sel.merge(
    km_diario[['TRACTORA', 'data', 'km']],
    on=['TRACTORA', 'data'],
    how='left'
)

# Verificar duplicatas (mesma matrícula, mesma data, múltiplos CODEUT)
duplicatas = df_merged[df_merged.duplicated(subset=['TRACTORA', 'data'], keep=False)]

if len(duplicatas) > 0:
    print(f"\n⚠ Encontradas {len(duplicatas):,} linhas com múltiplos CODEUT por matrícula/data")
    print("Distribuindo KM proporcionalmente...")
    
    # Distribuir KM
    for (tractor, data_val), group in df_merged.groupby(['TRACTORA', 'data']):
        if len(group) > 1 and group['km'].notna().any():
            km_total = group['km'].iloc[0]
            km_distribuido = km_total / len(group)
            df_merged.loc[(df_merged['TRACTORA'] == tractor) & (df_merged['data'] == data_val), 'km'] = km_distribuido

print(f"\n✓ KM adicionado a df_sel")
print(f"Linhas com KM: {df_merged['km'].notna().sum():,}")
print(f"Total colunas: {len(df_merged.columns)}")

# Atualizar df_sel
df_sel = df_merged.copy() 


# Verificar quantas linhas têm KM
print("="*80)
print("ANÁLISE DE KM")
print("="*80)

print(f"\nTotal linhas df_sel: {len(df_sel):,}")
print(f"Linhas com KM: {df_sel['km'].notna().sum():,}")
print(f"Linhas sem KM: {df_sel['km'].isna().sum():,}")
print(f"Percentagem com KM: {(df_sel['km'].notna().sum() / len(df_sel) * 100):.2f}%")

print(f"\n--- Estatísticas KM ---")
print(df_sel['km'].describe()) 